# Evaluate Qwen on perturbed data

Load the labeled perturbation dataset, select `Qwen/Qwen3.5-4B`, and retain only
original problems present in `train_allowed.parquet`, matching exact-text SHA-256
`problem_id` values. Use the repository's uv kernel.

`is_robust` and `max_drop` refer to the **original problem** for its model/effort
configuration. They are not measured outcomes for the generated variants.
Null labels remain unknown. Reference answers come from `solved.parquet`. Each variant is generated with
4-bit Qwen and scored by Math-Verify against its original reference answer.
Results are checkpointed after every completed fixed batch. This measures answer accuracy,
not accuracy at predicting the original robustness labels.

In [12]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from aimo_interp.preprocessing import generate_problem_id

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").is_file()
)
MODEL_ID = "Qwen/Qwen3.5-4B"
perturbed_path = REPO_ROOT / "data/prompt_perturbations/train_perturbed_labeled.parquet"
allowed_path = REPO_ROOT / "data/train-main-v2/train_allowed.parquet"

perturbed_df = pd.read_parquet(perturbed_path)
train_allowed_df = pd.read_parquet(allowed_path)
train_allowed_df["problem_id"] = generate_problem_id(train_allowed_df["problem"])

In [13]:
allowed_problem_ids = set(train_allowed_df["problem_id"])
qwen_perturbed_df = perturbed_df.loc[
    perturbed_df["model_id"].eq(MODEL_ID)
    & perturbed_df["problem_id"].isin(allowed_problem_ids)
].copy().reset_index(drop=True)
qwen_perturbed_df["is_robust"] = qwen_perturbed_df["is_robust"].astype("boolean")

assert qwen_perturbed_df["problem_id"].equals(
    generate_problem_id(qwen_perturbed_df["original_problem"])
), "Problem IDs do not match original problem text."
print(f"Filtered variants: {len(qwen_perturbed_df)}")
print(f"Unique original problems: {qwen_perturbed_df['problem_id'].nunique()}")
display(qwen_perturbed_df["perturbation_type"].value_counts().rename("variants"))
display(qwen_perturbed_df["is_robust"].value_counts(dropna=False).rename("variants_by_original_label"))

Filtered variants: 35
Unique original problems: 7


perturbation_type
rephrase    7
domain      7
rename      7
distract    7
typos       7
Name: variants, dtype: int64

is_robust
True     20
False    10
<NA>      5
Name: variants_by_original_label, dtype: Int64

In [14]:
qwen_perturbed_df.head()

,original_problem,perturbation_type,perturbed_problem,problem_id,model_id,reasoning_effort,is_robust,max_drop
0,The Fibonacci numbers are defined as follows: ...,rephrase,Let the Fibonacci sequence be given by $F_0 = ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
1,The Fibonacci numbers are defined as follows: ...,domain,A coding system assigns a sequence value $F_n$...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
2,The Fibonacci numbers are defined as follows: ...,rename,The Fibonacci numbers are defined as follows: ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
3,The Fibonacci numbers are defined as follows: ...,distract,The Fibonacci numbers are defined as follows: ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
4,The Fibonacci numbers are defined as follows: ...,typos,The Fibonacci numbers are defined as follows: ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01


In [15]:
qwen_perturbed_df

,original_problem,perturbation_type,perturbed_problem,problem_id,model_id,reasoning_effort,is_robust,max_drop
0,The Fibonacci numbers are defined as follows: ...,rephrase,Let the Fibonacci sequence be given by $F_0 = ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
1,The Fibonacci numbers are defined as follows: ...,domain,A coding system assigns a sequence value $F_n$...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
2,The Fibonacci numbers are defined as follows: ...,rename,The Fibonacci numbers are defined as follows: ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
3,The Fibonacci numbers are defined as follows: ...,distract,The Fibonacci numbers are defined as follows: ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
4,The Fibonacci numbers are defined as follows: ...,typos,The Fibonacci numbers are defined as follows: ...,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,Qwen/Qwen3.5-4B,default,True,-0.01
5,Find the sum of all real numbers $r$ such that...,rephrase,Determine the sum of all real numbers $r$ for ...,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,Qwen/Qwen3.5-4B,default,True,0.01
6,Find the sum of all real numbers $r$ such that...,distract,"For a classroom worksheet, the circle and para...",f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,Qwen/Qwen3.5-4B,default,True,0.01
7,Find the sum of all real numbers $r$ such that...,typos,Find the sum of all real numbers $r$ such that...,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,Qwen/Qwen3.5-4B,default,True,0.01
8,Find the sum of all real numbers $r$ such that...,rename,Find the sum of all real numbers $s$ such that...,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,Qwen/Qwen3.5-4B,default,True,0.01
9,Find the sum of all real numbers $r$ such that...,domain,"In a graphic design, a curved border follows t...",f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,Qwen/Qwen3.5-4B,default,True,0.01


In [16]:
qwen_perturbed_df.sort_values("problem_id").set_index("problem_id")[["perturbed_problem"]]

,perturbed_problem
problem_id,
26a263ad81138dbfb41647a1e55af77c0b8ab02d4b051fc23722a7f0c403535d,Find the sum of all integer bases $c>9$ for wh...
26a263ad81138dbfb41647a1e55af77c0b8ab02d4b051fc23722a7f0c403535d,Determine the sum of every integer base $b>9$ ...
26a263ad81138dbfb41647a1e55af77c0b8ab02d4b051fc23722a7f0c403535d,"During a number theory practice session, a stu..."
26a263ad81138dbfb41647a1e55af77c0b8ab02d4b051fc23722a7f0c403535d,A digital archive represents two record codes ...
26a263ad81138dbfb41647a1e55af77c0b8ab02d4b051fc23722a7f0c403535d,Find the sum of all integer bases $b>9$ for w...
7443669f7e9038ecbaa20040585b634dda36623e0129c5fb18d9e5cf895db73e,Define $S$ to be the sum of the infinite serie...
7443669f7e9038ecbaa20040585b634dda36623e0129c5fb18d9e5cf895db73e,Let $T$ denote the value of the infinite sum\n...
7443669f7e9038ecbaa20040585b634dda36623e0129c5fb18d9e5cf895db73e,"In a mathematical scoring model, stage $n$ con..."
7443669f7e9038ecbaa20040585b634dda36623e0129c5fb18d9e5cf895db73e,A mathematics club included the following prob...


## Join original solutions

The expected answer is used only by the scorer, never included in the model prompt.
This assumes each generated perturbation preserves the original answer.

In [17]:
solved_path = REPO_ROOT / "data/train-main-v2/solved.parquet"
solved_df = pd.read_parquet(solved_path)
assert solved_df["problem_id"].is_unique
qwen_evaluation_df = qwen_perturbed_df.merge(
    solved_df[["problem_id", "answer"]],
    on="problem_id", how="left", validate="many_to_one",
)
assert qwen_evaluation_df["answer"].notna().all(), "Missing reference solutions"
KEY_COLUMNS = ["model_id", "reasoning_effort", "problem_id", "perturbation_type"]
assert not qwen_evaluation_df.duplicated(KEY_COLUMNS).any()
display(qwen_evaluation_df[["problem_id", "perturbation_type", "answer"]].head())

,problem_id,perturbation_type,answer
0,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,rephrase,201
1,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,domain,201
2,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,rename,201
3,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,distract,201
4,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,typos,201


## Model and evaluation settings

Reuse an existing Qwen model in this kernel, or load it once. Thinking stays
enabled. An unfinished thinking trace is not treated as a final answer.
Math-Verify normalizes and compares the last boxed answer after `</think>`.

In [18]:
import importlib
import aimo_interp.model as model_helpers
import aimo_interp.evaluation as evaluation_helpers

# Refresh helper functions without reloading weights in an existing kernel.
importlib.reload(model_helpers)
importlib.reload(evaluation_helpers)
from aimo_interp.model import ANSWER_INSTRUCTIONS, load_model
from aimo_interp.evaluation import MathVerifyScorer, evaluate_batch

if "model" not in globals() or "tokenizer" not in globals():
    model, tokenizer = load_model(quantization_bits=4)
assert model.config._name_or_path == MODEL_ID
assert getattr(model, "is_loaded_in_4bit", False), "Expected the 4-bit model"

BATCH_SIZE = 1  # Try 2 initially on an 8 GB GPU; increase on the larger GPU.
MAX_NEW_TOKENS = 128 
ENABLE_THINKING = True
SCAFFOLDING_PROMPT = ANSWER_INSTRUCTIONS
RESULTS_PATH = REPO_ROOT / "data/evaluations/qwen_perturbed_fixed_batch_results.parquet"
METADATA_PATH = RESULTS_PATH.with_suffix(".metadata.json")
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

/home/suhas-shankar/Projects/AI_Research/AIMO_Interp/aimo-interp-investigate-black-box/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model': 'Qwen/Qwen3.5-4B', 'revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a', 'quantization_bits': 4, 'compute_dtype': 'torch.bfloat16'}
DeltaNet fast path: True
Loading tokenizer...
Loading and quantizing weights on GPU...


Loading weights: 100%|██████████| 723/723 [00:08<00:00, 85.44it/s] 

Loaded in 9.6 seconds
Model footprint: 3.01 GiB
CUDA allocated: 3.08 GiB
Parameter devices: ['cuda:0']


## Generate and score with fixed batches

Run `BATCH_SIZE` prompts together, waiting for the longest response before starting
another batch. Results are checkpointed after every completed batch. Rerunning
resumes completed rows. Changing inputs or settings requires a new output filename.

Per-row `elapsed_seconds` is the shared batch wall time, not the time at which
that individual response finished. `batch_tokens_per_second` measures aggregate
useful output throughput. Each row's token count excludes padding after EOS.

In [19]:
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

run_config = {
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "quantization": model.config.quantization_config.to_dict(),
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": BATCH_SIZE,
    "batching": "fixed",
    "enable_thinking": ENABLE_THINKING,
    "do_sample": False,
    "scaffolding_prompt": SCAFFOLDING_PROMPT,
    "versions": {p: version(p) for p in ["torch", "transformers", "bitsandbytes", "math-verify"]},
    "input_sha256": {str(p.relative_to(REPO_ROOT)): file_hash(p)
                     for p in [perturbed_path, allowed_path, solved_path]},
    "code_sha256": {name: file_hash(REPO_ROOT / "src/aimo_interp" / name)
                    for name in ["model.py", "evaluation.py"]},
    "scoring": "Math-Verify; last box in final response; unfinished thinking fails",
}
# Canonical JSON also makes any non-native config values serializable.
run_config = json.loads(json.dumps(run_config, default=str))
if RESULTS_PATH.exists():
    saved_metadata = json.loads(METADATA_PATH.read_text())
    assert saved_metadata["config"] == run_config, "Settings changed: choose a new RESULTS_PATH"
    results_df = pd.read_parquet(RESULTS_PATH)
else:
    METADATA_PATH.write_text(json.dumps({
        "created_at": datetime.now(timezone.utc).isoformat(), "config": run_config,
    }, indent=2) + "\n")
    results_df = pd.DataFrame()

records = results_df.to_dict("records")
completed = {tuple(row[k] for k in KEY_COLUMNS) for row in records}
assert isinstance(BATCH_SIZE, int) and BATCH_SIZE > 0
pending = [row for row in qwen_evaluation_df.to_dict("records")
           if tuple(row[k] for k in KEY_COLUMNS) not in completed]
next_batch_id = max((row["batch_id"] for row in records), default=-1) + 1
for offset in range(0, len(pending), BATCH_SIZE):
    batch = pending[offset:offset + BATCH_SIZE]
    scorers = [MathVerifyScorer(str(row["answer"])) for row in batch]
    results = evaluate_batch(
        [row["perturbed_problem"] for row in batch],
        SCAFFOLDING_PROMPT, model, scorers,
        tokenizer=tokenizer, max_new_tokens=MAX_NEW_TOKENS,
        enable_thinking=ENABLE_THINKING,
    )
    for row, result, scorer in zip(batch, results, scorers, strict=True):
        extracted = scorer.extract(result["final_response"])
        records.append({
            **row, **result, "batch_id": next_batch_id,
            "answer_extracted": bool(extracted),
            "extracted_answer": str(extracted[0]) if extracted else None,
        })
        completed.add(tuple(row[k] for k in KEY_COLUMNS))
    results_df = pd.DataFrame(records)
    temporary_path = RESULTS_PATH.with_suffix(".tmp.parquet")
    results_df.to_parquet(temporary_path, index=False)
    temporary_path.replace(RESULTS_PATH)
    print(f"{len(records)}/{len(qwen_evaluation_df)} | batch={next_batch_id} | "
          f"correct={sum(r['is_correct'] for r in results)}/{len(batch)} | "
          f"{results[0]['elapsed_seconds']:.1f}s | "
          f"{results[0]['batch_tokens_per_second']:.1f} tokens/s total", flush=True)
    next_batch_id += 1

print(f"Saved {len(results_df)} evaluations to {RESULTS_PATH}")

Generating batch of 1...
1/35 | batch=0 | correct=0/1 | 5.6s | 22.9 tokens/s total
Generating batch of 1...
2/35 | batch=1 | correct=0/1 | 4.3s | 29.8 tokens/s total
Generating batch of 1...
3/35 | batch=2 | correct=0/1 | 4.2s | 30.2 tokens/s total
Generating batch of 1...
4/35 | batch=3 | correct=0/1 | 4.2s | 30.2 tokens/s total
Generating batch of 1...
5/35 | batch=4 | correct=0/1 | 4.4s | 29.2 tokens/s total
Generating batch of 1...
6/35 | batch=5 | correct=0/1 | 4.9s | 26.1 tokens/s total
Generating batch of 1...
7/35 | batch=6 | correct=0/1 | 11.5s | 11.2 tokens/s total
Generating batch of 1...
8/35 | batch=7 | correct=0/1 | 8.8s | 14.6 tokens/s total
Generating batch of 1...
9/35 | batch=8 | correct=0/1 | 21.4s | 6.0 tokens/s total
Generating batch of 1...
10/35 | batch=9 | correct=0/1 | 21.9s | 5.9 tokens/s total
Generating batch of 1...
11/35 | batch=10 | correct=0/1 | 4.4s | 29.3 tokens/s total
Generating batch of 1...
12/35 | batch=11 | correct=0/1 | 4.3s | 30.0 tokens/s tota

## Answer accuracy

Unknown original robustness labels remain unknown; they do not prevent scoring
a variant against its reference answer. Review incorrect responses and any
perturbations that may have changed the mathematical answer.

In [20]:
results_df = pd.read_parquet(RESULTS_PATH)
print(f"Correct: {results_df['is_correct'].sum()}/{len(results_df)} "
      f"({results_df['is_correct'].mean():.1%})")
batch_seconds = results_df.drop_duplicates("batch_id")["elapsed_seconds"].sum()
print(f"Aggregate throughput: {results_df['generated_tokens'].sum() / batch_seconds:.1f} tokens/s")
print(f"Token limit reached: {results_df['hit_token_limit'].sum()}")
print(f"No final boxed answer extracted: {(~results_df['answer_extracted']).sum()}")
display(results_df.groupby("perturbation_type")["is_correct"].agg(["count", "sum", "mean"]))
display(results_df.groupby("is_robust", dropna=False)["is_correct"].agg(["count", "sum", "mean"]))
display(results_df[["problem_id", "perturbation_type", "answer", "extracted_answer", "is_correct", "hit_token_limit"]])

Correct: 0/35 (0.0%)
Aggregate throughput: 22.3 tokens/s
Token limit reached: 35
No final boxed answer extracted: 35


,count,sum,mean
perturbation_type,,,
distract,7,0,0.0
domain,7,0,0.0
rename,7,0,0.0
rephrase,7,0,0.0
typos,7,0,0.0


,count,sum,mean
is_robust,,,
True,20,0,0.0
NaN,5,0,0.0
False,10,0,0.0


,problem_id,perturbation_type,answer,extracted_answer,is_correct,hit_token_limit
0,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,rephrase,201,None,False,True
1,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,domain,201,None,False,True
2,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,rename,201,None,False,True
3,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,distract,201,None,False,True
4,d06c3e7f2d19047cd3277aa758fb2ec9c2f90726558b60...,typos,201,None,False,True
5,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,rephrase,50,None,False,True
6,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,distract,50,None,False,True
7,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,typos,50,None,False,True
8,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,rename,50,None,False,True
9,f3d7b78d243a1a500bf8f0ef91a3ecfff3ebc894a9f7c2...,domain,50,None,False,True
